In [1]:
!pip install jsonschema

In [2]:
import os
import re
import json
import joblib
import requests
import pandas as pd

from jsonschema import validate, ValidationError

In [3]:
model = joblib.load("best_model.pkl")

print("Model Loaded Successfully")

Model Loaded Successfully


In [15]:
import os
import requests

API_KEY = os.environ["LLM_API_KEY"]

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "google/gemma-4-26b-a4b-it:free",
    "messages": [
        {
            "role": "user",
            "content": "Say Hello"
        }
    ],
    "temperature": 0
}

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=payload
)

print("Status Code:", response.status_code)
print(response.text)

Status Code: 200

         

         

         
{"id":"gen-1783865100-VqlXwrBvVDMVA0jtUS3F","object":"chat.completion","created":1783865100,"model":"google/gemma-4-26b-a4b-it-20260403:free","provider":"Darkbloom","system_fingerprint":null,"service_tier":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"Hello! How can I help you today?","refusal":null,"reasoning":null}}],"usage":{"prompt_tokens":16,"completion_tokens":10,"total_tokens":26,"cost":0,"is_byok":false,"prompt_tokens_details":{"cached_tokens":0,"cache_write_tokens":0,"audio_tokens":0,"video_tokens":0},"cost_details":{"upstream_inference_cost":0.00000213,"upstream_inference_prompt_cost":4.8e-7,"upstream_inference_completions_cost":0.00000165},"completion_tokens_details":{"reasoning_tokens":0,"image_tokens":0,"audio_tokens":0}}}


In [16]:
import os
import requests

def call_llm(prompt, temperature=0):
    headers = {
        "Authorization": f"Bearer {os.environ['LLM_API_KEY']}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "google/gemma-4-26b-a4b-it:free",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature
    }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload
    )

    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]

In [17]:
print(call_llm("Say Hello"))

Hello! How can I help you today?


In [18]:
SYSTEM_PROMPT = """
You are an AI assistant that explains email spam predictions.

Rules:
1. Explain why the email is classified as Spam or Ham.
2. Mention important features such as spam score, links, exclamation marks and capital letters.
3. Keep the explanation under 100 words.
4. Never reveal personal information.
5. Return the response in JSON format.
"""

In [19]:
schema = {
    "type": "object",
    "properties": {
        "prediction": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "explanation": {
            "type": "string"
        }
    },
    "required": [
        "prediction",
        "confidence",
        "explanation"
    ]
}

In [21]:
import pandas as pd
import joblib

# Load model
model = joblib.load("best_model.pkl")

# Load cleaned dataset
df = pd.read_csv("cleaned_data.csv")

# Remove target column
X = df.drop(columns=["label"])

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [22]:
sample = X.iloc[[0]]

print(sample)

   email_id            subject  email_length  num_links  num_exclamations  \
0         1  Claim your reward         161.0        7.0                 2   

   capital_letter_percent  has_offer sender_domain  spam_score  
0                    29.0          1     promo.net         100  


In [25]:
print(model.feature_names_in_)

['email_id' 'email_length' 'num_links' 'num_exclamations'
 'capital_letter_percent' 'has_offer' 'subject_Claim your reward'
 'subject_Class reminder' 'subject_Click to verify'
 'subject_Exclusive deal' 'subject_Family dinner' 'subject_Free vacation'
 'subject_Interview schedule' 'subject_Invoice attached'
 'subject_Limited time offer' 'subject_Lottery winner'
 'subject_Monthly report' 'subject_Project meeting' 'subject_Team update'
 'subject_Urgent account update' 'subject_Win cash now'
 'sender_domain_gmail.com' 'sender_domain_offers.org'
 'sender_domain_outlook.com' 'sender_domain_promo.net'
 'sender_domain_yahoo.com']


In [26]:
df = pd.read_csv("cleaned_data.csv")

In [27]:
X = df.drop(columns=["label", "spam_score"])

In [28]:
X = pd.get_dummies(X, drop_first=True)

In [29]:
expected_cols = model.feature_names_in_

X = X.reindex(columns=expected_cols, fill_value=0)

In [30]:
sample = X.iloc[[0]]

prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0]

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: 1
Probability: [0.12423378 0.87576622]


In [31]:
import json

prompt = f"""
{SYSTEM_PROMPT}

The machine learning model predicted:

Prediction: Spam

Confidence: {probability[1]:.2f}

Email Features:

{sample.to_dict(orient='records')[0]}

Return ONLY valid JSON in this format:

{{
  "prediction":"Spam",
  "confidence":0.88,
  "explanation":"Explain why the model predicted spam."
}}
"""

response = call_llm(prompt)

print(response)

```json
{
  "prediction": "Spam",
  "confidence": 0.88,
  "explanation": "The email is classified as Spam due to a high confidence score of 0.88. Key indicators include a high number of links (7), a suspicious sender domain (promo.net), and a subject line ('Claim your reward') designed to entice the user. Additionally, the email contains a significant amount of capital letters (29%) and exclamation marks, which are common patterns in promotional or fraudulent content."
}
```


In [35]:
import json
from jsonschema import validate

# Remove markdown code block
clean_response = response.replace("```json", "").replace("```", "").strip()

# Convert to Python dictionary
result = json.loads(clean_response)

# Validate JSON
validate(instance=result, schema=schema)

print("✅ JSON Validation Successful!")
print(result)

✅ JSON Validation Successful!
{'prediction': 'Spam', 'confidence': 0.88, 'explanation': "The email is classified as Spam due to a high confidence score of 0.88. Key indicators include a high number of links (7), a suspicious sender domain (promo.net), and a subject line ('Claim your reward') designed to entice the user. Additionally, the email contains a significant amount of capital letters (29%) and exclamation marks, which are common patterns in promotional or fraudulent content."}


In [36]:
print("Temperature = 0")
print(call_llm(prompt, temperature=0))

print("\n=========================\n")

print("Temperature = 0.7")
print(call_llm(prompt, temperature=0.7))

Temperature = 0
```json
{
  "prediction": "Spam",
  "confidence": 0.88,
  "explanation": "The email is classified as Spam due to a high confidence score of 0.88. Key indicators include a high number of links (7), a suspicious sender domain (promo.net), and a subject line focused on rewards. Additionally, the email contains a significant amount of capital letters (29%) and exclamation marks, which are common patterns in promotional spam designed to create urgency."
}
```


Temperature = 0.7
```json
{
  "prediction": "Spam",
  "confidence": 0.88,
  "explanation": "The email is classified as Spam due to a high confidence score of 0.88. Key indicators include the presence of multiple links (7) and a suspicious sender domain (promo.net). The subject line 'Claim your reward' suggests promotional intent, and the high percentage of capital letters (29%) is common in spam. These features, combined with the promotional nature of the content, strongly suggest non-legitimate communication."
}
```


In [37]:
pii_prompt = """
User Email:
naveen@gmail.com

Phone:
9876543210

Explain the prediction but NEVER reveal the email address or phone number.
"""

print(call_llm(pii_prompt))

Based on the information provided, the prediction is based on the unique identifiers associated with the user profile. The analysis was conducted using the specific contact details provided in your input to identify patterns and determine the most likely outcome.


In [38]:
for i in range(3):
    sample = X.iloc[[i]]

    pred = model.predict(sample)[0]
    proba = model.predict_proba(sample)[0]

    print("=" * 50)
    print("Sample", i + 1)
    print("Prediction:", "Spam" if pred == 1 else "Ham")
    print("Confidence:", round(proba[1], 2))

Sample 1
Prediction: Spam
Confidence: 0.88
Sample 2
Prediction: Spam
Confidence: 0.98
Sample 3
Prediction: Spam
Confidence: 0.98


In [39]:
%%writefile requirements.txt
pandas
numpy
matplotlib
scikit-learn
joblib
requests
jsonschema

Writing requirements.txt


In [40]:
!ls

best_model.pkl	cleaned_data.csv  requirements.txt  sample_data
